# Leakage-Safe Temporal Target Encoding

Target encoding can expose a row's own label or future fraud outcomes unless history is processed strictly. This experiment uses timestamp-batched temporal OOF encoding inside TRAIN and frozen fit-only maps for later periods. External VALIDATION is evaluated once after TRAIN-CV selection; HOLDOUT is never assigned or accessed.

In [1]:
import gc
import json
import sys
import time
from pathlib import Path

import lightgbm as lgb
import mlflow
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fraud_detection.data import (
    load_train_identity,
    load_train_transaction,
    merge_train_data,
    split_data_chronologically,
    validate_training_data,
)
from fraud_detection.features import (
    BASE_TARGET_ENTITIES,
    COMPOSITE_TARGET_ENTITIES,
    TemporalTargetEncoder,
)
from fraud_detection.models import (
    evaluate_probabilities,
    make_expanding_time_folds,
    prepare_engineered_matrices,
)

notebook_started = time.perf_counter()

## Smoothing, temporal OOF methodology, and leakage audit

For entity `e`, `(fraud_count_e + 50 Ã— historical_global_prior) / (labeled_count_e + 50)` is calculated from labels at timestamps strictly earlier than the current row. All rows sharing a timestamp are encoded before that timestamp updates state. The earliest timestamp uses the fixed prior `0.5`, never the full TRAIN mean. Frozen evaluation maps fall back to the fit-derived global fraud rate for unseen values.

In [2]:
audit = pd.DataFrame(
    {
        "TransactionID": [1, 2, 3, 4],
        "TransactionDT": [100, 100, 200, 300],
        "card1": ["A", "B", "A", "A"],
        "card2": [1, 2, 1, 1],
        "addr1": [10, 20, 10, 10],
        "P_emaildomain": ["p1", "p2", "p1", "p1"],
        "R_emaildomain": ["r1", "r2", "r1", "r1"],
        "DeviceInfo": ["mobile", "desktop", "mobile", "mobile"],
        "ProductCD": ["W", "C", "W", "W"],
        "isFraud": [1, 0, 0, 1],
    }
)
audit_original = TemporalTargetEncoder().fit_transform_features(audit)
own_changed = audit.copy()
own_changed.loc[2, "isFraud"] = 1
own_result = TemporalTargetEncoder().fit_transform_features(own_changed)
future_changed = audit.copy()
future_changed.loc[3, "isFraud"] = 0
future_result = TemporalTargetEncoder().fit_transform_features(future_changed)
same_time_changed = audit.copy()
same_time_changed.loc[0, "isFraud"] = 0
same_time_result = TemporalTargetEncoder().fit_transform_features(same_time_changed)
frozen_encoder = TemporalTargetEncoder().fit(audit.iloc[:3])
evaluation = audit.iloc[[3]].copy()
evaluation_changed = evaluation.copy()
evaluation_changed["isFraud"] = 0
audit_results = pd.Series(
    {
        "own_target_independent": audit_original.loc[2].equals(own_result.loc[2]),
        "future_target_invariant": audit_original.iloc[:3].equals(future_result.iloc[:3]),
        "same_timestamp_isolated": audit_original.loc[1].equals(same_time_result.loc[1]),
        "evaluation_label_independent": frozen_encoder.transform_features(evaluation).equals(
            frozen_encoder.transform_features(evaluation_changed)
        ),
    },
    name="passed",
)
assert audit_results.all()
display(audit_results)

own_target_independent          True
future_target_invariant         True
same_timestamp_isolated         True
evaluation_label_independent    True
Name: passed, dtype: bool

## Entities and TRAIN-only temporal CV

In [3]:
raw_dir = PROJECT_ROOT / "data" / "raw"
transaction = load_train_transaction(raw_dir)
identity = load_train_identity(raw_dir)
merged = merge_train_data(transaction, identity)
split = split_data_chronologically(merged)
train = split.train
validate_training_data(train)
del transaction, identity, merged
gc.collect()

folds = make_expanding_time_folds(train, n_splits=3, initial_train_fraction=0.70)
fold_design = pd.DataFrame(
    [
        {
            "Fold": number,
            "Fit Rows": len(fold.train_indices),
            "Evaluation Rows": len(fold.evaluation_indices),
            "Fit DT Max": train.iloc[fold.train_indices]["TransactionDT"].max(),
            "Evaluation DT Min": train.iloc[fold.evaluation_indices]["TransactionDT"].min(),
            "Evaluation DT Max": train.iloc[fold.evaluation_indices]["TransactionDT"].max(),
        }
        for number, fold in enumerate(folds, start=1)
    ]
)
base_te_features = tuple(f"{entity}_target_encoded_fraud_rate" for entity in BASE_TARGET_ENTITIES)
composite_te_features = tuple(
    f"{entity}_target_encoded_fraud_rate" for entity in COMPOSITE_TARGET_ENTITIES
)
feature_sets = (
    "baseline",
    "baseline_plus_base_target_encoding",
    "baseline_plus_composite_target_encoding",
    "baseline_plus_all_target_encoding",
)
lightgbm_parameters = {
    "learning_rate": 0.1,
    "n_estimators": 400,
    "num_leaves": 63,
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}
display(fold_design)
display(pd.Series((*BASE_TARGET_ENTITIES, *COMPOSITE_TARGET_ENTITIES), name="entities"))
print("External validation status: not assigned or accessed before CV selection")
print("Final holdout status: not assigned or accessed")

,Fold,Fit Rows,Evaluation Rows,Fit DT Max,Evaluation DT Min,Evaluation DT Max
0,1,289364,41338,7135483,7135609,8121450
1,2,330702,41338,8121450,8121466,9247568
2,3,372040,41338,9247568,9247583,10437996


0                          card1
1                          card2
2                          addr1
3                  P_emaildomain
4                  R_emaildomain
5                     DeviceInfo
6              card1_card2_addr1
7      card1_addr1_p_emaildomain
8    card1_card2_addr1_productcd
9         card1_addr1_deviceinfo
Name: entities, dtype: object

External validation status: not assigned or accessed before CV selection
Final holdout status: not assigned or accessed


In [4]:
tracking_uri = f"sqlite:///{(PROJECT_ROOT / 'model_comparison.db').resolve().as_posix()}"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("fraud-detection-temporal-target-encoding")
cv_metrics = {name: [] for name in feature_sets}
feature_counts = {}
fit_seconds = {name: [] for name in feature_sets}
preparation_seconds = []
prepared_memory_mib = []

for fold_number, fold in enumerate(folds, start=1):
    fit_frame = train.iloc[fold.train_indices]
    evaluation_frame = train.iloc[fold.evaluation_indices]
    preparation_started = time.perf_counter()
    prepared = prepare_engineered_matrices(
        fit_frame,
        evaluation_frame,
        include_graph=False,
        include_composite=True,
        include_target_encoding=True,
        scaling_batch_rows=len(fit_frame),
    )
    preparation_seconds.append(time.perf_counter() - preparation_started)
    prepared_memory_mib.append((prepared.x_train.nbytes + prepared.x_evaluation.nbytes) / 1024**2)
    positions = {name: index for index, name in enumerate(prepared.feature_names)}
    baseline_names = tuple(
        name for name in prepared.feature_names if not name.endswith("_target_encoded_fraud_rate")
    )
    names_by_set = {
        "baseline": baseline_names,
        "baseline_plus_base_target_encoding": baseline_names + base_te_features,
        "baseline_plus_composite_target_encoding": (baseline_names + composite_te_features),
        "baseline_plus_all_target_encoding": (
            baseline_names + base_te_features + composite_te_features
        ),
    }
    indices_by_set = {
        name: np.asarray([positions[column] for column in names], dtype=np.int32)
        for name, names in names_by_set.items()
    }
    y_fit = fit_frame["isFraud"].astype("int8")
    y_evaluation = evaluation_frame["isFraud"].astype("int8")

    for feature_set in feature_sets:
        selected = indices_by_set[feature_set]
        fit_started = time.perf_counter()
        model = lgb.LGBMClassifier(**lightgbm_parameters)
        model.fit(prepared.x_train[:, selected], y_fit)
        probabilities = model.predict_proba(prepared.x_evaluation[:, selected])[:, 1]
        fit_seconds[feature_set].append(time.perf_counter() - fit_started)
        metrics = evaluate_probabilities(y_evaluation, probabilities)
        cv_metrics[feature_set].append(metrics)
        feature_counts[feature_set] = len(selected)
        print(f"fold={fold_number} feature_set={feature_set} PR-AUC={metrics.pr_auc:.6f}")
        del model, probabilities
    del prepared, indices_by_set, names_by_set
    gc.collect()

cv_rows = []
cv_run_ids = {}
encoded_counts = {
    "baseline": 0,
    "baseline_plus_base_target_encoding": len(base_te_features),
    "baseline_plus_composite_target_encoding": len(composite_te_features),
    "baseline_plus_all_target_encoding": (len(base_te_features) + len(composite_te_features)),
}
for feature_set in feature_sets:
    fold_pr_auc = [metric.pr_auc for metric in cv_metrics[feature_set]]
    fold_roc_auc = [metric.roc_auc for metric in cv_metrics[feature_set]]
    mean_pr_auc = float(np.mean(fold_pr_auc))
    std_pr_auc = float(np.std(fold_pr_auc, ddof=0))
    mean_roc_auc = float(np.mean(fold_roc_auc))
    with mlflow.start_run(run_name=f"temporal-te-cv-{feature_set}") as run:
        mlflow.set_tags(
            {
                "stage": "temporal-target-encoding",
                "dataset": "ieee-cis",
                "selection": "temporal-cv",
            }
        )
        mlflow.log_params(
            {
                "target_encoding_feature_set": feature_set,
                "encoded_entity_count": encoded_counts[feature_set],
                "alpha": 50.0,
                "feature_count": feature_counts[feature_set],
                **lightgbm_parameters,
            }
        )
        mlflow.log_metrics(
            {
                "fold_1_pr_auc": fold_pr_auc[0],
                "fold_2_pr_auc": fold_pr_auc[1],
                "fold_3_pr_auc": fold_pr_auc[2],
                "mean_cv_pr_auc": mean_pr_auc,
                "std_cv_pr_auc": std_pr_auc,
                "mean_cv_roc_auc": mean_roc_auc,
            }
        )
        cv_run_ids[feature_set] = run.info.run_id
    cv_rows.append(
        {
            "Feature Set": feature_set,
            "Encoded Entities": encoded_counts[feature_set],
            "Feature Count": feature_counts[feature_set],
            "Fold PR-AUC": tuple(fold_pr_auc),
            "Mean CV PR-AUC": mean_pr_auc,
            "Std CV PR-AUC": std_pr_auc,
            "Mean CV ROC-AUC": mean_roc_auc,
            "MLflow Run ID": cv_run_ids[feature_set],
        }
    )
cv_results = (
    pd.DataFrame(cv_rows).sort_values("Mean CV PR-AUC", ascending=False).reset_index(drop=True)
)
winner = cv_results.iloc[0]
display(cv_results)
print(f"TRAIN-CV winner: {winner['Feature Set']}")

2026/10/08 10:28:01 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection-temporal-target-encoding' does not exist. Creating a new experiment.


fold=1 feature_set=baseline PR-AUC=0.554229


fold=1 feature_set=baseline_plus_base_target_encoding PR-AUC=0.576419


fold=1 feature_set=baseline_plus_composite_target_encoding PR-AUC=0.596947


fold=1 feature_set=baseline_plus_all_target_encoding PR-AUC=0.599215


...

fold=3 feature_set=baseline_plus_base_target_encoding PR-AUC=0.612933


fold=3 feature_set=baseline_plus_composite_target_encoding PR-AUC=0.630908


fold=3 feature_set=baseline_plus_all_target_encoding PR-AUC=0.617945


,Feature Set,Encoded Entities,Feature Count,Fold PR-AUC,Mean CV PR-AUC,Std CV PR-AUC,Mean CV ROC-AUC,MLflow Run ID
0,baseline_plus_composite_target_encoding,4,467,"(0.5969474918067366, 0.5760601358100342, 0.630...",0.601305,0.022603,0.890932,730f6ac2d90a4e0caec77a9688f89641
1,baseline_plus_all_target_encoding,10,473,"(0.5992147729957521, 0.5757312034091924, 0.617...",0.597630,0.017270,0.891550,671aa5d734ef4073b8d17e966e8b43fe
2,baseline,0,463,"(0.5542290843797867, 0.5748513290293689, 0.625...",0.584875,0.029965,0.906202,ae23ee6afa0e44e8b42d7c69c0bdbf9b
3,baseline_plus_base_target_encoding,6,469,"(0.5764192272398987, 0.5413268324779132, 0.612...",0.576893,0.029235,0.895151,1d1e6123d8fc460eac7f0ed9c311b6b4


TRAIN-CV winner: baseline_plus_composite_target_encoding


## One external VALIDATION evaluation and conclusion

The winning TE subset is now fixed. Full TRAIN receives temporal OOF features, frozen maps are fitted using full TRAIN labels, and external VALIDATION receives only those frozen-map values.

In [5]:
validation = split.validation
final_started = time.perf_counter()
prepared_final = prepare_engineered_matrices(
    train,
    validation,
    include_graph=False,
    include_composite=True,
    include_target_encoding=True,
    scaling_batch_rows=len(train),
)
winning_name = str(winner["Feature Set"])
positions = {name: index for index, name in enumerate(prepared_final.feature_names)}
baseline_names = tuple(
    name for name in prepared_final.feature_names if not name.endswith("_target_encoded_fraud_rate")
)
final_names_by_set = {
    "baseline": baseline_names,
    "baseline_plus_base_target_encoding": baseline_names + base_te_features,
    "baseline_plus_composite_target_encoding": baseline_names + composite_te_features,
    "baseline_plus_all_target_encoding": (
        baseline_names + base_te_features + composite_te_features
    ),
}
selected = np.asarray(
    [positions[name] for name in final_names_by_set[winning_name]], dtype=np.int32
)
final_model = lgb.LGBMClassifier(**lightgbm_parameters)
final_model.fit(prepared_final.x_train[:, selected], train["isFraud"].astype("int8"))
validation_probabilities = final_model.predict_proba(prepared_final.x_evaluation[:, selected])[:, 1]
validation_metrics = evaluate_probabilities(
    validation["isFraud"].astype("int8"), validation_probabilities
)
final_seconds = time.perf_counter() - final_started
final_memory_mib = (prepared_final.x_train.nbytes + prepared_final.x_evaluation.nbytes) / 1024**2
with mlflow.start_run(run_name="temporal-te-final-winner") as run:
    mlflow.set_tags(
        {
            "stage": "temporal-target-encoding",
            "dataset": "ieee-cis",
            "selection": "temporal-cv",
            "final_refit": "true",
        }
    )
    mlflow.log_params(
        {
            "target_encoding_feature_set": winning_name,
            "encoded_entity_count": encoded_counts[winning_name],
            "alpha": 50.0,
            "feature_count": len(selected),
            **lightgbm_parameters,
        }
    )
    mlflow.log_metrics(
        {
            "validation_pr_auc": validation_metrics.pr_auc,
            "validation_roc_auc": validation_metrics.roc_auc,
            "validation_precision_at_0_50": validation_metrics.precision,
            "validation_recall_at_0_50": validation_metrics.recall,
            "validation_f1_at_0_50": validation_metrics.f1,
        }
    )
    final_run_id = run.info.run_id

previous_pr_auc = 0.595387
absolute_change = validation_metrics.pr_auc - previous_pr_auc
relative_change = absolute_change / previous_pr_auc
total_seconds = time.perf_counter() - notebook_started
mean_fit_seconds = {name: round(float(np.mean(values)), 1) for name, values in fit_seconds.items()}


def markdown_cv_table(frame):
    rows = [
        "| Feature set | TE entities | Features | Fold PR-AUCs | Mean | Std | Mean ROC-AUC |",
        "| --- | ---: | ---: | --- | ---: | ---: | ---: |",
    ]
    for _, row in frame.iterrows():
        fold_text = " / ".join(f"{value:.6f}" for value in row["Fold PR-AUC"])
        rows.append(
            f"| {row['Feature Set']} | {int(row['Encoded Entities'])} | "
            f"{int(row['Feature Count'])} | {fold_text} | {row['Mean CV PR-AUC']:.6f} | "
            f"{row['Std CV PR-AUC']:.6f} | {row['Mean CV ROC-AUC']:.6f} |"
        )
    return "\n".join(rows)


entity_lines = "\n".join(
    f"- `{entity}_target_encoded_fraud_rate`"
    for entity in (*BASE_TARGET_ENTITIES, *COMPOSITE_TARGET_ENTITIES)
)
audit_lines = "\n".join(
    f"- {name}: **{'PASS' if passed else 'FAIL'}**" for name, passed in audit_results.items()
)
run_lines = "\n".join(f"- **{name}:** `{run_id}`" for name, run_id in cv_run_ids.items())
change_word = "improvement" if absolute_change > 0 else "degradation"
report = f"""# Leakage-Safe Temporal Target Encoding

## Encoded entities and features

{entity_lines}

Exactly 10 target-encoded predictors are available: six base entities and four existing
composite identities. No support count or fraud count is exposed to the model.

## Formula, prior behavior, and missing values

`encoded_rate = (fraud_count + 50 * historical_global_prior) / (labeled_count + 50)`.
The earliest timestamp uses the explicit fixed prior **0.5**, because no historical labels exist.
Later TRAIN timestamps use the prior from strictly earlier labeled timestamps. Frozen evaluation
maps use the fit-derived global fraud rate for unseen entities. Base missing values use the
deterministic `<MISSING>` bucket; composites reuse length-prefixed keys and the `M` component.

## Temporal methodology and leakage audit

TRAIN features are timestamp-batched temporal OOF values. Rows at time `T` use only labels where
`TransactionDT < T`; an entire timestamp updates state only after every row at that timestamp is
encoded. Evaluation uses frozen fit-only maps and ignores evaluation labels.

{audit_lines}

## TRAIN-only temporal CV

{markdown_cv_table(cv_results)}

- Selected configuration: **{winning_name}**
- Selected feature count: **{len(selected)}**

## External validation

- PR-AUC: **{validation_metrics.pr_auc:.6f}**
- ROC-AUC: **{validation_metrics.roc_auc:.6f}**
- Precision at 0.50: **{validation_metrics.precision:.6f}**
- Recall at 0.50: **{validation_metrics.recall:.6f}**
- F1 at 0.50: **{validation_metrics.f1:.6f}**
- Confusion matrix: **{validation_metrics.confusion_matrix}**
- Previous preferred PR-AUC: **{previous_pr_auc:.6f}**
- Absolute {change_word}: **{absolute_change:+.6f}**
- Relative {change_word}: **{relative_change:+.2%}**

## Runtime and memory

- Fold preparation seconds: **{", ".join(f"{value:.1f}" for value in preparation_seconds)}**
- Mean model-fit seconds: **{json.dumps(mean_fit_seconds)}**
- Final TRAIN/VALIDATION workflow: **{final_seconds:.1f} seconds**
- Peak prepared matrix estimate: **{max([*prepared_memory_mib, final_memory_mib]):.1f} MiB**
- Total notebook time before report write: **{total_seconds:.1f} seconds**

## MLflow

- Experiment: `fraud-detection-temporal-target-encoding`
{run_lines}
- **Final winner:** `{final_run_id}`

## Production implications and limitations

Target-encoding maps may only be refreshed using historical labels available during retraining
or a controlled feature-state update. Label delay, state versioning, and timestamp-batch semantics
must be preserved in production. Alpha and the fixed earliest prior were not tuned.

External VALIDATION was not used for feature selection and was evaluated once. HOLDOUT was never
assigned, transformed, predicted, summarized, or evaluated. No weighting, graph features, tuning,
calibration, threshold optimization, ensembling, SHAP, or rules are included.
"""
report_path = PROJECT_ROOT / "reports" / "temporal_target_encoding_summary.md"
report_path.write_text(report, encoding="utf-8")
print(f"Winner: {winning_name}")
print(f"Validation PR-AUC: {validation_metrics.pr_auc:.6f}")
print(f"Absolute change: {absolute_change:+.6f}")
print(f"Relative change: {relative_change:+.2%}")
print(f"Final MLflow run: {final_run_id}")
print(f"Report: {report_path.relative_to(PROJECT_ROOT)}")
print("External validation used for selection: False")
print("Final holdout status: not assigned or accessed")

Winner: baseline_plus_composite_target_encoding
Validation PR-AUC: 0.612204
Absolute change: +0.016817
Relative change: +2.82%
Final MLflow run: 8ddc6a38eed74be2a071b43b98d3c188
Report: reports\temporal_target_encoding_summary.md
External validation used for selection: False
Final holdout status: not assigned or accessed
